# Continuous treatment in ModernDiD and R

This notebook is generated from the [Continuous treatment in ModernDiD and R](https://moderndid.readthedocs.io/en/stable/r_comparisons/r_continuous.html) page of the ModernDiD documentation.

This comparison checks whether the county employment analysis of the
fracking panel gives the same results in ModernDiD and R's `contdid`
package. We fit [`cont_did`](https://moderndid.readthedocs.io/en/stable/api/generated/didcont/moderndid.cont_did.html) in both languages and compare
the overall effects and the fitted curves at every score. Both calls use a
cubic spline without interior knots and compare treated counties with
counties that have not yet received treatment.

The overall ATT and ACRT estimates agree to floating-point precision. The dose
curves and every standard error differ because `contdid` 0.1.1 and the
`ptetools` 1.0.1 functions it calls depart in two ways from the method of
Callaway, Goodman-Bacon, and Sant'Anna (2025) that both packages implement.
They evaluate each cohort's fitted spline on a basis built from the grid's
endpoints rather than the basis it was fit on. Their influence functions also
scale the ACRT's cell terms by every county in the cell and drop or shrink the
term for estimated cohort weights. The sections below trace each gap to the R
code that produces it and show how large it is on this data.

R runs inside the Python session through `rpy2` and receives the same
observations. The section on [running the comparisons](https://moderndid.readthedocs.io/en/stable/r_comparisons/index.html#r-comparisons-running)
explains how to install the R packages and rerun the comparison locally.

> **What this comparison checks**
>
> The assertions hold the two overall estimates to the common tolerance and
> check that ModernDiD's curves do not move with the grid. Since R computes
> the dose curves and the influence functions differently, those values appear
> side by side without an assertion. Each language also draws its own random
> multipliers for the bootstrap standard errors.

In [ ]:
import subprocess
import sys

if "google.colab" in sys.modules:
    # Colab ships rpy2 3.5 as a single package. Uninstalling it first keeps pip from deleting
    # the files it shares with the split rpy2 3.6 packages.
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "rpy2"], check=True)
    # rpy2 supports Colab's R 4.6 from release 3.6.7 on.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "moderndid[all]==0.4.0", "rpy2>=3.6.7"], check=True)
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "r-cran-contdid", "r-cran-splines2"], check=True)
else:
    print("Outside Colab, set up R as https://moderndid.readthedocs.io/en/stable/r_comparisons/index.html#r-comparisons-running describes.")

In [ ]:
import numpy as np
import polars as pl
import rpy2.robjects as ro

import moderndid as did

## The same counties, years, and scores

The bundled fracking panel from [`load_fracking`](https://moderndid.readthedocs.io/en/stable/api/generated/data/moderndid.load_fracking.html) is the one
that the [continuous treatment example](https://moderndid.readthedocs.io/en/stable/user_guide/example_cont_did.html#example-cont-did) analyzes. It
contains 402 counties observed annually from 1990 through 2014. The outcome
`y` is already log total employment, `i` identifies counties, and `t` gives
the calendar year. The geological prospectivity score `d` stays fixed within
each county. `G` records the adoption year of the county's formation and
stays at zero throughout the panel for a coded-zero county.

> **How the scores were coded**
>
> Coded zeros include prospectivity scores that were originally missing. The
> scores were also constructed separately within each shale play. The
> [continuous treatment example](https://moderndid.readthedocs.io/en/stable/user_guide/example_cont_did.html#example-cont-did) explains why these
> coding and scale choices matter for interpreting the estimates.

The code keeps the five estimation columns of the loader's complete panel of
10,050 county-years. Before fitting, we encode the years 1990 through 2014
as periods 1 through 25 and give each positive adoption year the same
encoding. Untreated counties keep zero as their adoption code in both
languages. The original years stay available in the `calendar_year` and
`adoption_year` columns. Both implementations receive the same data with
every county-year still in the sample. The 20-score grid from 3.0 to 4.6
stays inside the range from 2.99 to 4.67 that every adoption cohort observes
and drops no county from estimation.

> **Encode calendar years consistently**
>
> The installed R implementation restores calendar-year labels for its
> estimated cohorts but keeps integer cohort codes in the data that computes
> the overall weights. Passing calendar years directly therefore stops R with
> a missing-weight error. Encoding both observation and adoption years as
> consecutive periods in both languages avoids that mismatch without changing
> the observations or the relative treatment timing.

In [ ]:
data = did.load_fracking().select("i", "t", "G", "d", "y")
first_year = data["t"].min()
data = data.with_columns(
    pl.col("t").alias("calendar_year"),
    pl.col("G").alias("adoption_year"),
    (pl.col("t") - first_year + 1).alias("t"),
    pl.when(pl.col("G") > 0).then(pl.col("G") - first_year + 1).otherwise(0).alias("G"),
)
dvals = np.linspace(3.0, 4.6, 20)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr

with ro.default_converter.context():
    importr("contdid")
with (ro.default_converter + pandas2ri.converter).context():
    ro.globalenv["panel"] = data.to_pandas()
    ro.globalenv["dvals"] = ro.FloatVector(dvals)

## Fit the same specification

The two calls print the Python result and then the R summary. Both evaluate
the curves at the same 20 scores and use 199 multiplier bootstrap draws for
pointwise uncertainty. Since post-adoption comparisons use the year before
adoption under either base period, the dose curves keep the post-adoption
target of the continuous treatment example.

Of the two packages, only ModernDiD reads the comparison-group and
base-period arguments that both calls pass. In `contdid` 0.1.1, `cont_did()`
never passes `control_group` or `base_period` on to `ptetools::pte()`. Its
subset function `cont_two_by_two_subset()` then falls back to its own
defaults, not-yet-treated comparisons and a varying base period.

> **`contdid` ignores both settings**
>
> A `contdid` call that asks for never-treated comparisons or a universal base
> period silently returns the same numbers as one that does not. The agreement
> on this page therefore holds only for not-yet-treated comparisons. Since only
> ModernDiD's estimates move when you ask for never-treated comparisons, report
> ModernDiD's numbers for that design.

In [ ]:
python_fit = did.cont_did(
    data,
    yname="y", tname="t", idname="i", gname="G", dname="d",
    target_parameter="level", aggregation="dose", treatment_type="continuous",
    dose_est_method="parametric", control_group="notyettreated", base_period="varying",
    degree=3, num_knots=0, dvals=dvals,
    biters=199, cband=False, random_state=42,
)
print(python_fit)

In [ ]:
with ro.default_converter.context():
    ro.r("set.seed(42)")
    r_fit = ro.r("""
        contdid::cont_did(
            data = panel,
            yname = "y", tname = "t", idname = "i", gname = "G", dname = "d",
            target_parameter = "level", aggregation = "dose", treatment_type = "continuous",
            dose_est_method = "parametric", control_group = "notyettreated", base_period = "varying",
            degree = 3, num_knots = 0, dvals = dvals,
            bstrap = TRUE, biters = 199, cband = FALSE
        )
    """)
    _ = ro.r["print"](ro.r["summary"](r_fit))

## Compare the reported values

The overall ATT averages the level effects over the treated score
distribution. The overall ACRT averages the fitted dose derivatives over the
same distribution. Reading it causally requires the stronger assumptions
described in [Difference-in-differences with a continuous treatment](https://moderndid.readthedocs.io/en/stable/background/didcont.html#background-didcont).

The next table extracts these quantities from the two fitted objects. Both
difference columns report Python minus R from unrounded results so that
small numerical gaps stay visible beside the reported values.

In [ ]:
with ro.default_converter.context():
    r_estimate = np.array([r_fit.rx2("overall_att")[0], r_fit.rx2("overall_acrt")[0]])
    r_se = np.array([r_fit.rx2("overall_att_se")[0], r_fit.rx2("overall_acrt_se")[0]])
python_estimate = np.array([python_fit.overall_att, python_fit.overall_acrt])
python_se = np.array([python_fit.overall_att_se, python_fit.overall_acrt_se])
overall = pd.DataFrame(
    {
        "Python estimate": python_estimate,
        "R estimate": r_estimate,
        "estimate difference": python_estimate - r_estimate,
        "Python SE": python_se,
        "R SE": r_se,
        "SE difference": python_se - r_se,
    },
    index=["Overall ATT", "Overall ACRT"],
)
overall

The two overall estimates agree to within 1e-16 of each other. The bootstrap
standard errors differ by 0.0012 for the ATT and by a factor of three for
the ACRT. The ATT gap is about the size that independent draws produce. The
[inference check](https://moderndid.readthedocs.io/en/stable/r_comparisons/r_continuous.html#compare-inference-before-bootstrap-draws) traces the ACRT
gap to the way `contdid` scales the ACRT influence function.

> **Independent bootstrap draws**
>
> Python and R use separate random number streams, even when both seeds
> are 42. With 199 draws, the standard error from one run varies by about
> 8 percent of its value from seed to seed. Two independent runs on the
> same influence function therefore often differ by 10 percent or more.

## Compare the dose curves

In the figure below, lines show the Python estimates and open circles show R
at the same scores. The two tables then list every estimate and standard
error beside its signed difference, first for the ATT curve and then for the
ACRT curve.

In [ ]:
with ro.default_converter.context():
    r_dose = np.asarray(r_fit.rx2("dose"))
    r_att = np.asarray(r_fit.rx2("att.d"))
    r_acrt = np.asarray(r_fit.rx2("acrt.d"))
    r_att_se = np.asarray(r_fit.rx2("att.d_se"))
    r_acrt_se = np.asarray(r_fit.rx2("acrt.d_se"))
np.testing.assert_array_equal(python_fit.dose, r_dose)

att_curve = pd.DataFrame({
    "dose": dvals,
    "Python ATT": python_fit.att_d,
    "R ATT": r_att,
    "ATT difference": python_fit.att_d - r_att,
    "Python SE": python_fit.att_d_se,
    "R SE": r_att_se,
    "SE difference": python_fit.att_d_se - r_att_se,
})
acrt_curve = pd.DataFrame({
    "dose": dvals,
    "Python ACRT": python_fit.acrt_d,
    "R ACRT": r_acrt,
    "ACRT difference": python_fit.acrt_d - r_acrt,
    "Python SE": python_fit.acrt_d_se,
    "R SE": r_acrt_se,
    "SE difference": python_fit.acrt_d_se - r_acrt_se,
})

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), dpi=200, constrained_layout=True)
for ax, name, table in zip(axes, ["ATT", "ACRT"], [att_curve, acrt_curve], strict=True):
    ax.plot(table["dose"], table[f"Python {name}"], label="Python", color="#176b87")
    ax.scatter(table["dose"], table[f"R {name}"], label="R", facecolors="none", edgecolors="#b7532c")
    ax.set(xlabel="Prospectivity score", ylabel=name)
    ax.legend(frameon=False)
plt.show()

display(att_curve)
display(acrt_curve)

### How R evaluates the curves

You can see in the figure that the curves part most at the top of the score
range. At a score of 4.6, ModernDiD puts the ATT at 0.057 and the ACRT at
0.063, against 0.111 and 0.319 in R. Across the grid, ModernDiD's ACRT curve
stays between 0.002 and 0.063. R's ACRT falls from 0.114 to −0.037 before it
climbs to 0.319.

Both packages fit each cohort's spline on a basis whose boundary knots are the
smallest and largest scores observed in that cohort. ModernDiD then evaluates
that fit at the grid scores on the same basis. In `contdid`, `cont_did_acrt`
instead applies the coefficients to `bs_grid`, a basis built on `dvals` alone
that takes its boundary knots from the grid's endpoints, 3.0 and 4.6. With
degree 3 and no interior knots, the basis consists of cubic Bernstein
polynomials on the interval between its boundary knots.

> **R's dose curves aren't the fitted curves**
>
> Since the same coefficients describe a different cubic on a different
> interval, R's ATT(d) and ACRT(d) are not the cohort's fitted dose response and
> its derivative, the curves that Callaway, Goodman-Bacon, and Sant'Anna (2025)
> estimate. Report ModernDiD's curves instead, since they evaluate each fit on
> the basis it was estimated on and keep their values when you change the
> grid.

The two lines of the installed `cont_did_acrt` that build the fitting basis
`bs` and the evaluation basis `bs_grid` appear below. The table after them
sets each cohort's observed range beside the grid boundaries that R uses.
The [splines2 documentation](https://wwenjie.org/splines2/reference/bspline)
explains how the boundary knots default to the input's range. The [contdid
source](https://github.com/bcallaway11/contdid/blob/master/R/cont_did.R) on
GitHub holds the level and derivative calculations. In `ptetools`,
`process_dose_gt` builds the influence functions of the curves on the same
grid basis (`R/process_dose_gt.R`, lines 85 to 88).

In [ ]:
with ro.default_converter.context():
    r_source = list(ro.r("deparse(body(contdid::cont_did_acrt))"))
    for index, line in enumerate(r_source):
        if "bs <- as.data.frame" in line or "bs_grid <- as.data.frame" in line:
            print(line.strip())
            print(r_source[index + 1].strip())
    grid_bounds = np.asarray(ro.r('attr(splines2::bSpline(dvals, degree = 3), "Boundary.knots")'))

cohort_doses = data.filter(pl.col("G") > 0).to_pandas().groupby("adoption_year")["d"]
pd.DataFrame({
    "observed minimum": cohort_doses.min(),
    "observed maximum": cohort_doses.max(),
    "R grid minimum": grid_bounds[0],
    "R grid maximum": grid_bounds[1],
})

### Repeat the comparison on a narrower grid

We repeat both calls with the outer three scores removed from each end of
`dvals`. All observations, estimator settings, and interior knots stay the
same. The table reports the largest change in each curve at the 14 scores
that both grids share. A curve that is the fitted dose response does not
change at those scores, whatever grid you pass. The overall summaries use
the observed scores and do not depend on the grid.

In [ ]:
narrow_dvals = dvals[3:-3]
narrow_python = did.cont_did(
    data,
    yname="y", tname="t", idname="i", gname="G", dname="d",
    target_parameter="level", aggregation="dose", treatment_type="continuous",
    dose_est_method="parametric", control_group="notyettreated", base_period="varying",
    degree=3, num_knots=0, dvals=narrow_dvals,
    biters=199, cband=False, random_state=42,
)

In [ ]:
with ro.default_converter.context():
    ro.globalenv["narrow_dvals"] = ro.FloatVector(narrow_dvals)
    ro.r("set.seed(42)")
    narrow_r = ro.r("""
        contdid::cont_did(
            data = panel,
            yname = "y", tname = "t", idname = "i", gname = "G", dname = "d",
            target_parameter = "level", aggregation = "dose", treatment_type = "continuous",
            dose_est_method = "parametric", control_group = "notyettreated", base_period = "varying",
            degree = 3, num_knots = 0, dvals = narrow_dvals,
            bstrap = TRUE, biters = 199, cband = FALSE
        )
    """)

In [ ]:
with ro.default_converter.context():
    narrow_r_dose = np.asarray(narrow_r.rx2("dose"))
    narrow_r_att = np.asarray(narrow_r.rx2("att.d"))
    narrow_r_acrt = np.asarray(narrow_r.rx2("acrt.d"))
np.testing.assert_array_equal(narrow_python.dose, narrow_dvals)
np.testing.assert_array_equal(narrow_r_dose, narrow_dvals)

grid_changes = pd.DataFrame(
    {
        "Python maximum change at shared scores": [
            np.max(np.abs(narrow_python.att_d - python_fit.att_d[3:-3])),
            np.max(np.abs(narrow_python.acrt_d - python_fit.acrt_d[3:-3])),
        ],
        "R maximum change at shared scores": [
            np.max(np.abs(narrow_r_att - r_att[3:-3])),
            np.max(np.abs(narrow_r_acrt - r_acrt[3:-3])),
        ],
    },
    index=["ATT", "ACRT"],
)
np.testing.assert_allclose(grid_changes["Python maximum change at shared scores"], 0, atol=1e-12)
grid_changes

Both of ModernDiD's curves keep the same value at every score that the two
grids share. Although only the grid changed, R's ATT moves by up to 0.059 at
those scores and its ACRT by up to 0.309.


## Compare inference before bootstrap draws

An influence function (IF) describes each county's contribution to sampling
uncertainty before any multiplier draws. The next table applies
`sqrt(sum(IF**2)) / n` to the influence functions that both fits return. For
each curve it reports the largest of the 20 values on each side and the
largest gap between them. Of the four rows, the overall ACRT shows the
largest relative gap.

In [ ]:
with ro.default_converter.context():
    r_overall_att_if = np.asarray(r_fit.rx2("overall_att_inffunc")).reshape(-1)
    r_overall_acrt_if = np.asarray(r_fit.rx2("overall_acrt_inffunc")).reshape(-1)
    r_att_curve_if = np.asarray(r_fit.rx2("att.d_inffunc"))
    r_acrt_curve_if = np.asarray(r_fit.rx2("acrt.d_inffunc"))


def influence_se(values):
    """Return the standard error that each column of an influence function implies."""
    assert np.isfinite(values).all()
    assert values.shape[0] == data["i"].n_unique()
    return np.sqrt(np.sum(values**2, axis=0)) / values.shape[0]


inference_rows = []
for name, python_if, r_if in (
    ("Overall ATT", python_fit.overall_att_inf_func, r_overall_att_if),
    ("Overall ACRT", python_fit.overall_acrt_inf_func, r_overall_acrt_if),
    ("ATT curve", python_fit.att_d_inf_func, r_att_curve_if),
    ("ACRT curve", python_fit.acrt_d_inf_func, r_acrt_curve_if),
):
    python_se, r_se = influence_se(python_if), influence_se(r_if)
    inference_rows.append({
        "quantity": name,
        "largest Python SE implied by IF": np.max(python_se),
        "largest R SE implied by IF": np.max(r_se),
        "largest absolute SE difference": np.max(np.abs(python_se - r_se)),
    })
pd.DataFrame(inference_rows)

ModernDiD's influence function gives the overall ACRT a standard error of
0.0099, against 0.0029 in R. In `contdid`, `cont_did_acrt` returns each
cell's ACRT influence function on the cell's treated counties alone.
`ptetools` then multiplies every cell's influence function by n/n1 with n1
the number of counties in the whole cell (`R/pte.R`, line 137). Since the
ACRT averages over treated counties, ModernDiD scales by n over the number
of treated counties instead. R's factor shrinks each cell's term by the
treated share of the cell. In the first post-adoption year of the 2001
cohort, that share is 34 of 402 counties.

`process_dose_gt` also averages the cell influence functions with the cohort
weights held fixed (`R/process_dose_gt.R`, line 74). Since those weights are
estimated cohort shares, ModernDiD adds the term that their estimation
contributes. The scaling accounts for most of the gap and the weight term
for the rest.

> **R's ACRT interval is too narrow**
>
> R prints an overall ACRT interval from 0.0372 to 0.0511 around the same
> estimate, about a third as wide as ModernDiD's interval from 0.0236 to 0.0648.
> The scaling above, rather than any extra precision, makes R's interval
> narrower. Report ModernDiD's interval, since its influence function scales
> each cell by the treated counties that the ACRT averages over and adds the
> term for estimated cohort weights.

The overall ATT values differ only slightly, 0.00942 in ModernDiD against
0.00925 in R. Before it builds the term for estimated cohort shares,
`pte_aggte` rescales the shares to sum to one (`R/pte_aggte.R`, lines 66 and
67). That term then comes out multiplied by the share of counties ever
treated, 329 of 402 here.

For the curves, `process_dose_gt` builds the influence functions on the grid
basis and combines the cells with fixed weights (lines 123 and 148). Line 118
also negates the comparison counties' term in ATT(d). Since the stored binary
ATT influence function already gives that term its sign, the negation flips
it. On this data the flip changes the curve standard errors by at most 0.1
percent. The largest curve values, 0.0120 against 0.0158 for ATT(d) and 0.0259
against 0.0762 for ACRT(d), sit at the ends of the grid.

The last check requires the two overall estimates to agree within
`1e-6 + 1e-5 * abs(R estimate)`. Its table lists the largest gap in each
comparison.

In [ ]:
np.testing.assert_allclose(overall["Python estimate"], overall["R estimate"], rtol=1e-5, atol=1e-6)
pd.DataFrame(
    {
        "largest absolute estimate difference": [
            overall["estimate difference"].abs().max(),
            att_curve["ATT difference"].abs().max(),
            acrt_curve["ACRT difference"].abs().max(),
        ],
        "largest absolute SE difference": [
            overall["SE difference"].abs().max(),
            att_curve["SE difference"].abs().max(),
            acrt_curve["SE difference"].abs().max(),
        ],
    },
    index=["overall estimates", "ATT curve", "ACRT curve"],
)

The curve rows repeat the grid-basis gaps from the figure. Their
standard-error gaps combine independent draws with the influence-function
differences traced above.

## Overall effects in an event study

Although this page fits dose curves, the two packages also report different
overall effects in an event study. With `aggregation="eventstudy"`,
[`cont_did`](https://moderndid.readthedocs.io/en/stable/api/generated/didcont/moderndid.cont_did.html) follows Callaway and Sant'Anna (2021) and reports
the average of the event-study effects from the start of treatment on as
`overall_att`. `contdid` computes that average too but returns its group
aggregation as the top-level `overall_att`. Since both are defensible
summaries, the two numbers differ by definition.

## Versions used

These versions of Python, R, and the packages produced the output shown on
this page.

In [ ]:
from importlib.metadata import version
from platform import python_version

with ro.default_converter.context():
    r_version = ro.r("R.version.string")[0]
    contdid_version = ro.r('as.character(packageVersion("contdid"))')[0]
    ptetools_version = ro.r('as.character(packageVersion("ptetools"))')[0]
pd.DataFrame({
    "component": ["Python", "ModernDiD", "rpy2", "R", "contdid", "ptetools"],
    "version": [python_version(), version("moderndid"), version("rpy2"), r_version, contdid_version, ptetools_version],
})